# 30-Day Mortality Rate Prediction for Acute Myocardial Infarction (AMI)

## Complete Analysis Pipeline: CRISP-DM Methodology

**Project Goal:** Predict 30-day mortality rates following heart attacks using US CMS hospital data.

**Dataset:** Centers for Medicare & Medicaid Services (CMS) Hospital Compare data including:
- Medicare spending per patient
- Hospital general information (type, ownership, location)
- Unplanned hospital visits metrics
- Complications and death rates

**Target Variable:** `MORT_30_AMI` - 30-day mortality rate (%) for acute myocardial infarction patients

**Analysis Approach:** 
- Data integration from 4 CMS datasets
- Exploratory data analysis with visualizations
- Feature engineering and selection
- Multiple regression models with hyperparameter tuning
- Model evaluation and deployment

## Business Questions

This analysis addresses five critical questions for healthcare stakeholders:

### 1. Which hospital characteristics most strongly predict 30-day AMI mortality rates?
- **Stakeholder:** Hospital administrators, quality improvement teams
- **Value:** Identify modifiable factors to reduce mortality

### 2. How do mortality rates vary by hospital type, ownership, and geographic location?
- **Stakeholder:** Policy makers, healthcare researchers
- **Value:** Understand structural and regional disparities

### 3. Can we accurately predict mortality rates using quality metrics and spending data?
- **Stakeholder:** CMS, insurers, hospital networks
- **Value:** Develop risk-adjusted performance benchmarks

### 4. Which features provide the most actionable insights for hospital improvement?
- **Stakeholder:** Clinical leadership, operations teams
- **Value:** Prioritize interventions with highest impact

### 5. What patterns exist in the relationship between hospital spending and mortality outcomes?
- **Stakeholder:** Healthcare economists, payers
- **Value:** Optimize resource allocation and reimbursement models

---






### Notebook Structure
1. Setup & Imports
2. Data Loading & Integration
3. Exploratory Data Analysis (EDA)
4. Feature Engineering & Selection
5. Baseline Model Development
6. Hyperparameter Tuning
7. Final Model Selection & Persistence
8. Business Questions Answers
9. Summary & Recommendations

In [ ]:

#SETUP: Import Required Libraries


import os
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# Configure matplotlib for better visualizations
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['font.size'] = 10


# UTILITY FUNCTIONS


def calculate_regression_metrics(y_true, y_pred):
 
    return {
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'R2': float(r2_score(y_true, y_pred))
    }


def plot_predictions_vs_actual(y_true, y_pred, model_name, color='steelblue'):

    plt.figure(figsize=(6, 6))
    plt.scatter(y_true, y_pred, alpha=0.6, color=color, edgecolors='white', linewidth=0.5)
    
    # Add perfect prediction line
    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    
    plt.xlabel('Actual 30-Day Mortality Rate (%)', fontsize=11)
    plt.ylabel('Predicted 30-Day Mortality Rate (%)', fontsize=11)
    plt.title(f'{model_name}: Actual vs Predicted', fontsize=13, fontweight='bold')
    plt.legend(loc='upper left')
    plt.tight_layout()
    plt.show()


def plot_residuals(y_true, y_pred, model_name, color='steelblue'):

    residuals = y_true - y_pred
    
    plt.figure(figsize=(7, 4))
    plt.hist(residuals, bins=30, color=color, edgecolor='white', alpha=0.7)
    plt.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    plt.xlabel('Residual (Actual - Predicted)', fontsize=11)
    plt.ylabel('Frequency', fontsize=11)
    plt.title(f'{model_name}: Residual Distribution', fontsize=13, fontweight='bold')
    plt.legend()
    plt.tight_layout()
    plt.show()


print("✓ Libraries loaded successfully")
print(f"✓ Working directory: {os.getcwd()}")
print("✓ Utility functions defined")

---

# CRISP-DM Phase 2: Data Understanding


In [ ]:

# DATA LOADING



NOTEBOOK_DIR = os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR) if 'notebooks' in NOTEBOOK_DIR else NOTEBOOK_DIR
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')


data_files = {
    'medicare_spending': os.path.join(DATA_DIR, 'Medicare_Hospital_Spending_Per_Patient-Hospital.csv'),
    'hospital_info': os.path.join(DATA_DIR, 'Hospital_General_Information.csv'),
    'unplanned_visits': os.path.join(DATA_DIR, 'Unplanned_Hospital_Visits-Hospital.csv'),
    'complications': os.path.join(DATA_DIR, 'Complications_and_Deaths-Hospital.csv')
}


print("Checking data files...")
for name, path in data_files.items():
    if os.path.exists(path):
        print(f"  ✓ {name}: Found")
    else:
        raise FileNotFoundError(f"Missing required file: {path}")


print("\nLoading datasets...")
medicare_spending_df = pd.read_csv(data_files['medicare_spending'], low_memory=False)
hospital_info_df = pd.read_csv(data_files['hospital_info'], low_memory=False)
unplanned_visits_df = pd.read_csv(data_files['unplanned_visits'], low_memory=False)
complications_df = pd.read_csv(data_files['complications'], low_memory=False)

print(f"  ✓ Medicare Spending: {medicare_spending_df.shape}")
print(f"  ✓ Hospital Info: {hospital_info_df.shape}")
print(f"  ✓ Unplanned Visits: {unplanned_visits_df.shape}")
print(f"  ✓ Complications: {complications_df.shape}")


# DATA CLEANING FUNCTIONS

def clean_facility_id(series):
   
    
    series = series.astype(str).str.replace(r'\D+', '', regex=True)
  
    series = series.replace('', pd.NA)
    
    series = pd.to_numeric(series, errors='coerce')
    return series.astype('Int64').astype(str)


def pivot_to_latest_measures(df, value_column='Score'):

    
    df_clean = df.dropna(subset=['Facility ID', 'Measure ID']).copy()
    
    
    if 'End Date' in df_clean.columns:
        latest_indices = df_clean.groupby(['Facility ID', 'Measure ID'])['End Date'].idxmax()
        df_latest = df_clean.loc[latest_indices]
    else:
        
        latest_indices = df_clean.groupby(['Facility ID', 'Measure ID']).head(1).index
        df_latest = df_clean.loc[latest_indices]
    
    
    pivoted = df_latest.pivot(index='Facility ID', columns='Measure ID', values=value_column)
    
    return pivoted


# CLEAN FACILITY IDs ACROSS ALL DATASETS


print("\nStandardizing Facility IDs...")
for df in [medicare_spending_df, hospital_info_df, unplanned_visits_df, complications_df]:
    df['Facility ID'] = clean_facility_id(df['Facility ID'])


# PARSE NUMERIC AND DATE COLUMNS

print("Parsing Score and Date columns...")
for df in [medicare_spending_df, unplanned_visits_df, complications_df]:
    if 'Score' in df.columns:
        df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
    if 'End Date' in df.columns:
        df['End Date'] = pd.to_datetime(df['End Date'], errors='coerce')


# PIVOT TIME-SERIES DATA TO WIDE FORMAT


print("\nPivoting datasets to wide format...")
medicare_spending_pivot = pivot_to_latest_measures(medicare_spending_df)
unplanned_visits_pivot = pivot_to_latest_measures(unplanned_visits_df)
complications_pivot = pivot_to_latest_measures(complications_df)

print(f"  ✓ Medicare Spending pivoted: {medicare_spending_pivot.shape}")
print(f"  ✓ Unplanned Visits pivoted: {unplanned_visits_pivot.shape}")
print(f"  ✓ Complications pivoted: {complications_pivot.shape}")

# CREATE UNIFIED DATASET


print("\nCreating unified dataset...")


hospital_static_df = hospital_info_df.drop_duplicates(subset=['Facility ID']).set_index('Facility ID')


unified_df = hospital_static_df.join(
    medicare_spending_pivot, how='outer'
).join(
    unplanned_visits_pivot, how='outer'
).join(
    complications_pivot, how='outer'
)

print(f"\n✓ Unified dataset created: {unified_df.shape[0]} hospitals, {unified_df.shape[1]} features")
print(f"✓ Target variable 'MORT_30_AMI' present: {'MORT_30_AMI' in unified_df.columns}")


print("\nSample of unified dataset:")
unified_df.head(3)

---

## Step 2 — Exploratory Data Analysis (EDA)


In [ ]:

# EXPLORATORY DATA ANALYSIS


TARGET_VARIABLE = 'MORT_30_AMI'


# 1. DATA OVERVIEW


print("="*70)
print("DATA OVERVIEW")
print("="*70)
print(f"Total hospitals: {unified_df.shape[0]:,}")
print(f"Total features: {unified_df.shape[1]:,}")


numeric_columns = unified_df.select_dtypes(include=['number']).columns.tolist()
categorical_columns = [col for col in unified_df.columns if col not in numeric_columns]

print(f"\nNumeric features: {len(numeric_columns)}")
print(f"Categorical features: {len(categorical_columns)}")


# 2. MISSING DATA ANALYSIS


print("\n" + "="*70)
print("MISSING DATA ANALYSIS")
print("="*70)

missing_percentage = unified_df.isna().mean().sort_values(ascending=False)

print("\nTop 10 features with highest missing data:")
for feature, pct in missing_percentage.head(10).items():
    print(f"  {feature:40s} {pct*100:5.1f}%")

# Visualize missing data
plt.figure(figsize=(12, 5))
missing_percentage.head(25).plot(kind='bar', color='coral', edgecolor='white')
plt.title('Percentage of Missing Values by Feature (Top 25)', fontsize=13, fontweight='bold')
plt.xlabel('Feature', fontsize=11)
plt.ylabel('Missing Percentage', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, 1)
plt.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='50% Threshold')
plt.legend()
plt.tight_layout()
plt.show()


# 3. TARGET VARIABLE ANALYSIS


print("\n" + "="*70)
print(f"TARGET VARIABLE: {TARGET_VARIABLE}")
print("="*70)

if TARGET_VARIABLE in unified_df.columns:
    target_values = pd.to_numeric(unified_df[TARGET_VARIABLE], errors='coerce').dropna()
    
    print(f"\nHospitals with {TARGET_VARIABLE} data: {len(target_values):,} ({len(target_values)/len(unified_df)*100:.1f}%)")
    print(f"Missing {TARGET_VARIABLE}: {unified_df[TARGET_VARIABLE].isna().sum():,} ({unified_df[TARGET_VARIABLE].isna().sum()/len(unified_df)*100:.1f}%)")
    
    print("\nDistribution Statistics:")
    print(target_values.describe())
    
   
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 2, 1)
    plt.hist(target_values, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    plt.axvline(target_values.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {target_values.mean():.2f}%')
    plt.axvline(target_values.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {target_values.median():.2f}%')
    plt.xlabel('30-Day Mortality Rate (%)', fontsize=11)
    plt.ylabel('Number of Hospitals', fontsize=11)
    plt.title(f'{TARGET_VARIABLE} Distribution', fontsize=12, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 2, 2)
    plt.boxplot(target_values, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='steelblue'),
                medianprops=dict(color='red', linewidth=2),
                whiskerprops=dict(color='steelblue'),
                capprops=dict(color='steelblue'))
    plt.ylabel('30-Day Mortality Rate (%)', fontsize=11)
    plt.title(f'{TARGET_VARIABLE} Box Plot', fontsize=12, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    

    # 4. CORRELATION ANALYSIS

    
    print("\n" + "="*70)
    print("CORRELATION ANALYSIS WITH TARGET")
    print("="*70)

    correlation_df = unified_df[numeric_columns].copy()
    correlation_df[TARGET_VARIABLE] = pd.to_numeric(correlation_df[TARGET_VARIABLE], errors='coerce')
    
    target_correlations = correlation_df.corr(numeric_only=True)[TARGET_VARIABLE].dropna()
    target_correlations = target_correlations.sort_values(key=lambda x: x.abs(), ascending=False)
    
    print("\nTop 15 features by absolute correlation with target:")
    for i, (feature, corr) in enumerate(target_correlations.head(15).items(), 1):
        direction = "positive" if corr > 0 else "negative"
        print(f"  {i:2d}. {feature:40s} {corr:+.4f} ({direction})")
    
 
    top_features = target_correlations.index[1:21]  
    
    if len(top_features) > 0:
    
        top_features_df = correlation_df[list(top_features)].dropna()
        correlation_matrix = top_features_df.corr(numeric_only=True)
        
        
        plt.figure(figsize=(10, 8))
        im = plt.imshow(correlation_matrix, cmap='coolwarm', vmin=-1, vmax=1, aspect='auto')
        
        plt.colorbar(im, label='Correlation Coefficient')
        plt.title('Correlation Heatmap: Top 20 Features Correlated with Target', 
                 fontsize=12, fontweight='bold', pad=20)
        
      
        plt.xticks(range(len(top_features)), top_features, rotation=90, fontsize=8, ha='right')
        plt.yticks(range(len(top_features)), top_features, fontsize=8)
        
        
        for i in range(len(top_features)):
            for j in range(len(top_features)):
                text_color = 'white' if abs(correlation_matrix.iloc[i, j]) > 0.5 else 'black'
                plt.text(j, i, f'{correlation_matrix.iloc[i, j]:.2f}',
                        ha='center', va='center', color=text_color, fontsize=6)
        
        plt.tight_layout()
        plt.show()
    
else:
    print(f"\n⚠ Target variable '{TARGET_VARIABLE}' not found in dataset!")

---

## Step 3 — Feature Assessment


In [ ]:

# FEATURE ASSESSMENT


print("="*70)
print("CATEGORICAL FEATURE CARDINALITY")
print("="*70)

categorical_cardinality = unified_df.select_dtypes(exclude=['number']).nunique().sort_values(ascending=False)

print("\nTop 10 categorical features by unique values:")
for feature, count in categorical_cardinality.head(10).items():
    print(f"  {feature:45s} {count:5,d} unique values")


# IDENTIFY POTENTIAL LEAKAGE COLUMNS


print("\n" + "="*70)
print("POTENTIAL DATA LEAKAGE ASSESSMENT")
print("="*70)


footnote_columns = [col for col in unified_df.columns if 'footnote' in col.lower()]

print(f"\nFound {len(footnote_columns)} footnote columns (potential leakage risk):")
for col in footnote_columns[:10]:  # Show first 10
    print(f"  - {col}")

if len(footnote_columns) > 10:
    print(f"  ... and {len(footnote_columns) - 10} more")

print("\n⚠ Note: Footnote columns often indicate data quality issues with the")
print("  corresponding measure and may be derived from the target variable itself.")
print("  These should be excluded from modeling to prevent data leakage.")


# FEATURE TYPE SUMMARY


print("\n" + "="*70)
print("FEATURE TYPE SUMMARY")
print("="*70)

print(f"\nTotal features: {len(unified_df.columns)}")
print(f"  - Numeric: {len(numeric_columns)}")
print(f"  - Categorical: {len(categorical_columns)}")
print(f"  - High cardinality (>1000): {(categorical_cardinality > 1000).sum()}")
print(f"  - Footnote columns: {len(footnote_columns)}")

---

# CRISP-DM Phase 3: Data Preparation



In [ ]:

# STEP 1: FILTER TO HOSPITALS WITH TARGET DATA


print("="*70)
print("DATA PREPARATION FOR MACHINE LEARNING")
print("="*70)


ml_df = unified_df[~unified_df[TARGET_VARIABLE].isna()].copy()

print(f"\nFiltering to hospitals with {TARGET_VARIABLE} data:")
print(f"  Original: {len(unified_df):,} hospitals")
print(f"  With target: {len(ml_df):,} hospitals ({len(ml_df)/len(unified_df)*100:.1f}%)")
print(f"  Removed: {len(unified_df) - len(ml_df):,} hospitals")


# STEP 2: DROP HIGH-MISSINGNESS FEATURES


print("\n" + "="*70)
print("DROPPING HIGH-MISSINGNESS FEATURES")
print("="*70)


missing_fraction = ml_df.isna().mean()


high_missing_columns = missing_fraction[missing_fraction > 0.5].index.tolist()
if TARGET_VARIABLE in high_missing_columns:
    high_missing_columns.remove(TARGET_VARIABLE)

print(f"\nDropping {len(high_missing_columns)} features with >50% missing data:")
for col in high_missing_columns[:10]: 
    print(f"  - {col:50s} ({missing_fraction[col]*100:.1f}% missing)")
if len(high_missing_columns) > 10:
    print(f"  ... and {len(high_missing_columns) - 10} more")


ml_df = ml_df.drop(columns=high_missing_columns)

print(f"\nRemaining features: {ml_df.shape[1]}")


# STEP 3: SEPARATE NUMERIC AND CATEGORICAL FEATURES


print("\n" + "="*70)
print("FEATURE TYPE SEPARATION")
print("="*70)


numeric_feature_cols = ml_df.select_dtypes(include=['number']).columns.tolist()
if TARGET_VARIABLE in numeric_feature_cols:
    numeric_feature_cols.remove(TARGET_VARIABLE)


candidate_categorical = ['Hospital Type', 'Hospital Ownership', 'Emergency Services', 'State']
categorical_feature_cols = [col for col in candidate_categorical if col in ml_df.columns]

print(f"\nNumeric features for modeling: {len(numeric_feature_cols)}")
print(f"Categorical features for encoding: {len(categorical_feature_cols)}")
print(f"\nSelected categorical features:")
for col in categorical_feature_cols:
    unique_count = ml_df[col].nunique()
    print(f"  - {col:30s} ({unique_count} categories)")


# STEP 4: IMPUTE MISSING VALUES


print("\n" + "="*70)
print("IMPUTING MISSING VALUES")
print("="*70)


if numeric_feature_cols:
    numeric_imputer = SimpleImputer(strategy='median')
    ml_df[numeric_feature_cols] = numeric_imputer.fit_transform(ml_df[numeric_feature_cols])
    print(f"  ✓ Numeric features imputed with median")
else:
    numeric_imputer = None
    print(f"  ! No numeric features to impute")

if categorical_feature_cols:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    ml_df[categorical_feature_cols] = categorical_imputer.fit_transform(ml_df[categorical_feature_cols])
    print(f"  ✓ Categorical features imputed with mode")
else:
    categorical_imputer = None
    print(f"  ! No categorical features to impute")


# STEP 5: ENCODE CATEGORICAL VARIABLES


print("\n" + "="*70)
print("ENCODING CATEGORICAL VARIABLES")
print("="*70)


if categorical_feature_cols:
    categorical_encoded_df = pd.get_dummies(
        ml_df[categorical_feature_cols], 
        drop_first=True,
        dtype=float
    )
    print(f"  ✓ Created {categorical_encoded_df.shape[1]} binary indicator variables")
else:
    categorical_encoded_df = pd.DataFrame(index=ml_df.index)
    print(f"  ! No categorical features to encode")


# STEP 6: SCALE NUMERIC FEATURES


print("\n" + "="*70)
print("SCALING NUMERIC FEATURES")
print("="*70)


if numeric_feature_cols:
    scaler = StandardScaler()
    numeric_scaled_df = pd.DataFrame(
        scaler.fit_transform(ml_df[numeric_feature_cols]),
        columns=numeric_feature_cols,
        index=ml_df.index
    )
    print(f"  ✓ Scaled {len(numeric_feature_cols)} numeric features (StandardScaler)")
else:
    numeric_scaled_df = pd.DataFrame(index=ml_df.index)
    scaler = None
    print(f"  ! No numeric features to scale")


# STEP 7: CREATE FINAL FEATURE MATRIX AND TARGET VECTOR


print("\n" + "="*70)
print("FINAL DATASET ASSEMBLY")
print("="*70)


X_full = pd.concat([numeric_scaled_df, categorical_encoded_df], axis=1)


y_full = pd.to_numeric(ml_df[TARGET_VARIABLE]).values

print(f"\n✓ Final feature matrix (X): {X_full.shape}")
print(f"  - Hospitals: {X_full.shape[0]:,}")
print(f"  - Features: {X_full.shape[1]:,}")
print(f"    • Numeric (scaled): {numeric_scaled_df.shape[1]}")
print(f"    • Categorical (encoded): {categorical_encoded_df.shape[1]}")

print(f"\n✓ Target vector (y): {y_full.shape}")
print(f"  - Mean: {y_full.mean():.2f}%")
print(f"  - Std: {y_full.std():.2f}%")
print(f"  - Range: [{y_full.min():.2f}%, {y_full.max():.2f}%]")

print("\n" + "="*70)
print("DATA PREPARATION COMPLETE")
print("="*70)

---

# CRISP-DM Phase 4: Modeling



In [ ]:

# TRAIN-TEST SPLIT


print("="*70)
print("BASELINE MODEL DEVELOPMENT")
print("="*70)


X_train, X_test, y_train, y_test = train_test_split(
    X_full, y_full, 
    test_size=0.2, 
    random_state=42
)

print(f"\nData Split:")
print(f"  Training set: {X_train.shape[0]:,} hospitals ({X_train.shape[0]/len(X_full)*100:.0f}%)")
print(f"  Test set:     {X_test.shape[0]:,} hospitals ({X_test.shape[0]/len(X_full)*100:.0f}%)")

# MODEL 1: LINEAR REGRESSION


print("\n" + "="*70)
print("MODEL 1: LINEAR REGRESSION")
print("="*70)

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)


y_pred_linear = linear_model.predict(X_test)


linear_metrics = calculate_regression_metrics(y_test, y_pred_linear)

print("\nLinear Regression Performance:")
print(f"  RMSE: {linear_metrics['RMSE']:.4f}")
print(f"  MAE:  {linear_metrics['MAE']:.4f}")
print(f"  R²:   {linear_metrics['R2']:.4f}")


# MODEL 2: RANDOM FOREST (BASELINE)


print("\n" + "="*70)
print("MODEL 2: RANDOM FOREST (BASELINE)")
print("="*70)


rf_baseline_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    verbose=0
)
rf_baseline_model.fit(X_train, y_train)


y_pred_rf_baseline = rf_baseline_model.predict(X_test)


rf_baseline_metrics = calculate_regression_metrics(y_test, y_pred_rf_baseline)

print("\nRandom Forest (Baseline) Performance:")
print(f"  RMSE: {rf_baseline_metrics['RMSE']:.4f}")
print(f"  MAE:  {rf_baseline_metrics['MAE']:.4f}")
print(f"  R²:   {rf_baseline_metrics['R2']:.4f}")


# DIAGNOSTIC VISUALIZATIONS


print("\n" + "="*70)
print("DIAGNOSTIC VISUALIZATIONS")
print("="*70)

print("\nGenerating diagnostic plots...")


plot_predictions_vs_actual(y_test, y_pred_linear, 'Linear Regression', color='steelblue')
plot_residuals(y_test, y_pred_linear, 'Linear Regression', color='steelblue')


plot_predictions_vs_actual(y_test, y_pred_rf_baseline, 'Random Forest (Baseline)', color='forestgreen')
plot_residuals(y_test, y_pred_rf_baseline, 'Random Forest (Baseline)', color='forestgreen')


# BASELINE COMPARISON


print("\n" + "="*70)
print("BASELINE MODEL COMPARISON")
print("="*70)

comparison_df = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest (Baseline)'],
    'RMSE': [linear_metrics['RMSE'], rf_baseline_metrics['RMSE']],
    'MAE': [linear_metrics['MAE'], rf_baseline_metrics['MAE']],
    'R²': [linear_metrics['R2'], rf_baseline_metrics['R2']]
})

print("\n", comparison_df.to_string(index=False))

best_baseline = 'Linear Regression' if linear_metrics['RMSE'] < rf_baseline_metrics['RMSE'] else 'Random Forest'
print(f"\n✓ Best baseline model by RMSE: {best_baseline}")

---

# CRISP-DM Phase 5: Evaluation


In [ ]:

# FEATURE SELECTION: TOP 200 BY CORRELATION


print("="*70)
print("FEATURE SELECTION")
print("="*70)


feature_correlations = pd.Series(index=X_train.columns, dtype=float)

for col in X_train.columns:
    try:
        correlation = np.corrcoef(X_train[col], y_train)[0, 1]
        feature_correlations[col] = correlation
    except Exception:
        feature_correlations[col] = np.nan


top_200_features = (
    feature_correlations.abs()
    .fillna(0.0)
    .sort_values(ascending=False)
    .head(200)
    .index.tolist()
)

print(f"\nFeature selection results:")
print(f"  Original features: {len(X_train.columns)}")
print(f"  Selected features: {len(top_200_features)}")
print(f"\nTop 10 most correlated features:")
for i, feat in enumerate(feature_correlations.abs().sort_values(ascending=False).head(10).index, 1):
    corr_val = feature_correlations[feat]
    print(f"  {i:2d}. {feat:50s} {corr_val:+.4f}")


X_train_reduced = X_train[top_200_features].copy()
X_test_reduced = X_test[top_200_features].copy()


X_train_tune, X_val_tune, y_train_tune, y_val_tune = train_test_split(
    X_train_reduced, y_train,
    test_size=0.25,
    random_state=42
)

print(f"\nData splits for tuning:")
print(f"  Training: {X_train_tune.shape[0]:,} hospitals")
print(f"  Validation: {X_val_tune.shape[0]:,} hospitals")
print(f"  Test (held-out): {X_test_reduced.shape[0]:,} hospitals")


# MODEL 1: ELASTICNET TUNING


print("\n" + "="*70)
print("TUNING ELASTICNET")
print("="*70)

elasticnet_param_grid = [
    {'alpha': 0.01, 'l1_ratio': 0.2},
    {'alpha': 0.01, 'l1_ratio': 0.5},
    {'alpha': 0.01, 'l1_ratio': 0.8},
    {'alpha': 0.1, 'l1_ratio': 0.2},
    {'alpha': 0.1, 'l1_ratio': 0.5},
    {'alpha': 0.1, 'l1_ratio': 0.8},
    {'alpha': 1.0, 'l1_ratio': 0.2},
    {'alpha': 1.0, 'l1_ratio': 0.5},
    {'alpha': 1.0, 'l1_ratio': 0.8},
]

best_en_params = None
best_en_rmse = np.inf

print("Testing hyperparameter combinations...")
for params in elasticnet_param_grid:
    model = ElasticNet(**params, max_iter=3000, random_state=42)
    model.fit(X_train_tune, y_train_tune)
    y_pred_val = model.predict(X_val_tune)
    rmse = np.sqrt(mean_squared_error(y_val_tune, y_pred_val))
    
    if rmse < best_en_rmse:
        best_en_rmse = rmse
        best_en_params = params

print(f"\n✓ Best ElasticNet parameters: {best_en_params}")
print(f"  Validation RMSE: {best_en_rmse:.4f}")


final_elasticnet_model = ElasticNet(**best_en_params, max_iter=3000, random_state=42)
final_elasticnet_model.fit(X_train_reduced, y_train)

y_pred_en_test = final_elasticnet_model.predict(X_test_reduced)
elasticnet_metrics = calculate_regression_metrics(y_test, y_pred_en_test)

print(f"\nElasticNet Test Set Performance:")
print(f"  RMSE: {elasticnet_metrics['RMSE']:.4f}")
print(f"  MAE:  {elasticnet_metrics['MAE']:.4f}")
print(f"  R²:   {elasticnet_metrics['R2']:.4f}")


# MODEL 2: RANDOM FOREST TUNING


print("\n" + "="*70)
print("TUNING RANDOM FOREST")
print("="*70)

rf_param_grid = [
    {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 1},
    {'n_estimators': 300, 'max_depth': None, 'min_samples_leaf': 2},
    {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 1},
    {'n_estimators': 300, 'max_depth': 12, 'min_samples_leaf': 2},
    {'n_estimators': 600, 'max_depth': None, 'min_samples_leaf': 1},
    {'n_estimators': 600, 'max_depth': None, 'min_samples_leaf': 2},
    {'n_estimators': 600, 'max_depth': 12, 'min_samples_leaf': 1},
    {'n_estimators': 600, 'max_depth': 12, 'min_samples_leaf': 2},
]

best_rf_params = None
best_rf_rmse = np.inf

print("Testing hyperparameter combinations...")
for params in rf_param_grid:
    model = RandomForestRegressor(**params, max_features='sqrt', random_state=42, n_jobs=-1)
    model.fit(X_train_tune, y_train_tune)
    y_pred_val = model.predict(X_val_tune)
    rmse = np.sqrt(mean_squared_error(y_val_tune, y_pred_val))
    
    if rmse < best_rf_rmse:
        best_rf_rmse = rmse
        best_rf_params = params

best_rf_params['max_features'] = 'sqrt'  

print(f"\n✓ Best Random Forest parameters: {best_rf_params}")
print(f"  Validation RMSE: {best_rf_rmse:.4f}")


final_rf_model = RandomForestRegressor(**best_rf_params, random_state=42, n_jobs=-1)
final_rf_model.fit(X_train_reduced, y_train)

y_pred_rf_test = final_rf_model.predict(X_test_reduced)
rf_tuned_metrics = calculate_regression_metrics(y_test, y_pred_rf_test)

print(f"\nRandom Forest Test Set Performance:")
print(f"  RMSE: {rf_tuned_metrics['RMSE']:.4f}")
print(f"  MAE:  {rf_tuned_metrics['MAE']:.4f}")
print(f"  R²:   {rf_tuned_metrics['R2']:.4f}")


# MODEL 3: GRADIENT BOOSTING TUNING


print("\n" + "="*70)
print("TUNING GRADIENT BOOSTING")
print("="*70)

gbm_param_grid = [
    {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 2},
    {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 3},
    {'n_estimators': 300, 'learning_rate': 0.1, 'max_depth': 2},
    {'n_estimators': 300, 'learning_rate': 0.1, 'max_depth': 3},
    {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 2},
    {'n_estimators': 600, 'learning_rate': 0.05, 'max_depth': 3},
    {'n_estimators': 600, 'learning_rate': 0.1, 'max_depth': 2},
    {'n_estimators': 600, 'learning_rate': 0.1, 'max_depth': 3},
]

best_gbm_params = None
best_gbm_rmse = np.inf

print("Testing hyperparameter combinations...")
for params in gbm_param_grid:
    model = GradientBoostingRegressor(**params, random_state=42)
    model.fit(X_train_tune, y_train_tune)
    y_pred_val = model.predict(X_val_tune)
    rmse = np.sqrt(mean_squared_error(y_val_tune, y_pred_val))
    
    if rmse < best_gbm_rmse:
        best_gbm_rmse = rmse
        best_gbm_params = params

print(f"\n✓ Best Gradient Boosting parameters: {best_gbm_params}")
print(f"  Validation RMSE: {best_gbm_rmse:.4f}")


final_gbm_model = GradientBoostingRegressor(**best_gbm_params, random_state=42)
final_gbm_model.fit(X_train_reduced, y_train)

y_pred_gbm_test = final_gbm_model.predict(X_test_reduced)
gbm_metrics = calculate_regression_metrics(y_test, y_pred_gbm_test)

print(f"\nGradient Boosting Test Set Performance:")
print(f"  RMSE: {gbm_metrics['RMSE']:.4f}")
print(f"  MAE:  {gbm_metrics['MAE']:.4f}")
print(f"  R²:   {gbm_metrics['R2']:.4f}")


# MODEL COMPARISON VISUALIZATION


print("\n" + "="*70)
print("TUNED MODEL COMPARISON")
print("="*70)

comparison_tuned_df = pd.DataFrame({
    'Model': ['ElasticNet', 'Random Forest', 'Gradient Boosting'],
    'RMSE': [elasticnet_metrics['RMSE'], rf_tuned_metrics['RMSE'], gbm_metrics['RMSE']],
    'MAE': [elasticnet_metrics['MAE'], rf_tuned_metrics['MAE'], gbm_metrics['MAE']],
    'R²': [elasticnet_metrics['R2'], rf_tuned_metrics['R2'], gbm_metrics['R2']]
}).sort_values('RMSE')

print("\nTuned Model Performance (sorted by RMSE):")
print(comparison_tuned_df.to_string(index=False))


plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_en_test, alpha=0.5, label='ElasticNet', color='purple', s=30)
plt.scatter(y_test, y_pred_rf_test, alpha=0.5, label='Random Forest', marker='x', color='green', s=30)
plt.scatter(y_test, y_pred_gbm_test, alpha=0.5, label='Gradient Boosting', marker='^', color='orange', s=30)

min_val = min(y_test.min(), y_pred_en_test.min(), y_pred_rf_test.min(), y_pred_gbm_test.min())
max_val = max(y_test.max(), y_pred_en_test.max(), y_pred_rf_test.max(), y_pred_gbm_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

plt.xlabel('Actual 30-Day Mortality Rate (%)', fontsize=12)
plt.ylabel('Predicted 30-Day Mortality Rate (%)', fontsize=12)
plt.title('Parity Plot: All Tuned Models (Top 200 Features)', fontsize=14, fontweight='bold')
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# FEATURE IMPORTANCE ANALYSIS


print("\n" + "="*70)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*70)


en_coefficients = final_elasticnet_model.coef_
en_coef_order = np.argsort(np.abs(en_coefficients))[::-1][:20]

plt.figure(figsize=(10, 6))
plt.barh(range(20)[::-1], en_coefficients[en_coef_order][::-1], color='mediumpurple', edgecolor='white')
plt.yticks(range(20)[::-1], X_train_reduced.columns[en_coef_order][::-1], fontsize=9)
plt.xlabel('Coefficient Value', fontsize=11)
plt.title('ElasticNet: Top 20 Features by Coefficient Magnitude', fontsize=12, fontweight='bold')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()


rf_importances = final_rf_model.feature_importances_
rf_importance_order = np.argsort(rf_importances)[::-1][:20]

plt.figure(figsize=(10, 6))
plt.barh(range(20)[::-1], rf_importances[rf_importance_order][::-1], color='forestgreen', edgecolor='white')
plt.yticks(range(20)[::-1], X_train_reduced.columns[rf_importance_order][::-1], fontsize=9)
plt.xlabel('Feature Importance (Gini)', fontsize=11)
plt.title('Random Forest: Top 20 Features by Importance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n✓ Hyperparameter tuning complete")

---

# CRISP-DM Phase 6: Deployment



In [ ]:

# SELECT BEST MODEL


print("="*70)
print("FINAL MODEL SELECTION")
print("="*70)


all_models = {
    'ElasticNet': {
        'model': final_elasticnet_model,
        'metrics': elasticnet_metrics,
        'features': top_200_features,
        'params': best_en_params
    },
    'RandomForest': {
        'model': final_rf_model,
        'metrics': rf_tuned_metrics,
        'features': top_200_features,
        'params': best_rf_params
    },
    'GradientBoosting': {
        'model': final_gbm_model,
        'metrics': gbm_metrics,
        'features': top_200_features,
        'params': best_gbm_params
    }
}


final_model_name = min(all_models, key=lambda k: all_models[k]['metrics']['RMSE'])
final_model_obj = all_models[final_model_name]['model']
final_model_metrics = all_models[final_model_name]['metrics']
final_model_features = all_models[final_model_name]['features']
final_model_params = all_models[final_model_name]['params']

print(f"\n✓ Selected Best Model: {final_model_name}")
print(f"\nHyperparameters: {final_model_params}")
print(f"\nTest Set Performance:")
print(f"  RMSE: {final_model_metrics['RMSE']:.4f}")
print(f"  MAE:  {final_model_metrics['MAE']:.4f}")
print(f"  R²:   {final_model_metrics['R2']:.4f}")


# SAVE MODEL ARTIFACTS


print("\n" + "="*70)
print("SAVING DEPLOYMENT ARTIFACTS")
print("="*70)


PKL_DIR = os.path.join(PROJECT_ROOT, 'pkl')
os.makedirs(PKL_DIR, exist_ok=True)


model_filename = os.path.join(PKL_DIR, f'{final_model_name.lower()}_final.pkl')
joblib.dump(final_model_obj, model_filename)
print(f"  ✓ Saved model: {model_filename}")

if numeric_imputer:
    numeric_imputer_file = os.path.join(PKL_DIR, 'num_imputer.pkl')
    joblib.dump(numeric_imputer, numeric_imputer_file)
    print(f"  ✓ Saved numeric imputer: {numeric_imputer_file}")

if categorical_imputer:
    categorical_imputer_file = os.path.join(PKL_DIR, 'cat_imputer.pkl')
    joblib.dump(categorical_imputer, categorical_imputer_file)
    print(f"  ✓ Saved categorical imputer: {categorical_imputer_file}")

if scaler:
    scaler_file = os.path.join(PKL_DIR, 'scaler.pkl')
    joblib.dump(scaler, scaler_file)
    print(f"  ✓ Saved scaler: {scaler_file}")


feature_list_file = os.path.join(PKL_DIR, 'top200_feature_list.json')
with open(feature_list_file, 'w') as f:
    json.dump(final_model_features, f, indent=2)
print(f"  ✓ Saved feature list: {feature_list_file}")





print("\n" + "="*70)
print("INFERENCE EXAMPLE")
print("="*70)

sample_X = X_test_reduced.iloc[:5]
sample_y_actual = y_test[:5]
sample_y_pred = final_model_obj.predict(sample_X)

print("\nSample Predictions:")
print("-" * 70)
print(f"{'Hospital Index':<20} {'Actual (%)':<15} {'Predicted (%)':<15} {'Error':<10}")
print("-" * 70)

for i, (actual, predicted) in enumerate(zip(sample_y_actual, sample_y_pred)):
    error = predicted - actual
    error_sign = '+' if error >= 0 else ''
    print(f"{i+1:<20} {actual:<15.2f} {predicted:<15.2f} {error_sign}{error:.2f}")

print("-" * 70)
print(f"Sample RMSE: {np.sqrt(mean_squared_error(sample_y_actual, sample_y_pred)):.4f}")
print(f"Sample MAE:  {mean_absolute_error(sample_y_actual, sample_y_pred):.4f}")

print("\n" + "="*70)
print("MODEL DEPLOYMENT ARTIFACTS SAVED")
print("="*70)


---

## References

**Data Source:**  
Centers for Medicare & Medicaid Services (CMS). Hospital Compare Data. Retrieved from https://data.cms.gov/

**Methodology:**  
Chapman, P., et al. (2000). CRISP-DM 1.0: Step-by-step data mining guide. SPSS Inc.

**Libraries:**  
- pandas: McKinney, W. (2010). Data Structures for Statistical Computing in Python.
- scikit-learn: Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. JMLR 12, pp. 2825-2830.

